# Forward Kinematics

The forward kinematics is calculated using the Sympy library.


In [43]:
import sympy as sm
import sympy.physics.mechanics as me
import numpy as np

# 1. Define Time and Dynamic Symbols
t = sm.symbols('t')
q1,q2,q3,q4,q5 = me.dynamicsymbols('q1:6')  # This creates q1(t), q2(t), q3(t), q4(t), q5(t)

# 2. Reference Frames
N = me.ReferenceFrame('N')  # World
B = me.ReferenceFrame('B')  # Base
S = me.ReferenceFrame('S')  # Shoulder
U = me.ReferenceFrame('U')  # Upper Arm
L = me.ReferenceFrame('L')  # Lower Arm
W = me.ReferenceFrame('W')  # Wrist
G = me.ReferenceFrame('G')  # Gripper


# 3. Rotations
B.orient_body_fixed(N, [0, 0, sm.pi], 'XYZ')
S.orient_body_fixed(B, [0, 0, q1], 'XYZ')
U.orient_body_fixed(S, [0, -sm.pi/2, q2], 'XYZ')
L.orient_body_fixed(U, [0, 0, q3], 'XYZ')
W.orient_body_fixed(L, [0, 0, sm.pi/2 + q4], 'XYZ')
G.orient_body_fixed(W, [0, -sm.pi/2, q5], 'XYZ')

# 4. Translations
r_B_S = 0*B.x - 0.0452*B.y + 0.0165*B.z
r_S_U = 0*S.x - 0.0306*S.y + 0.1025*S.z
r_U_L = 0.11257*U.x - 0.028*U.y + 0*U.z
r_L_W = 0.0052*L.x - 0.1349*L.y + 0*L.z
r_W_G = -0.0601*W.x + 0*W.y + 0*W.z
r_G_end = 0*G.x + 0*G.y + 0.075*G.z

# 5. Calculate End-effector Position
r_N_End = r_B_S + r_S_U + r_U_L + r_L_W + r_W_G + r_G_end

# 6. Express the result in the World Frame (N)
pos_vector = r_N_End.express(N).to_matrix(N).simplify()

# To get LaTeX code
#print(sm.latex(pos_vector))
pos_vector

Matrix([
[                                                               (0.0052*sin(q2(t) + q3(t)) + 0.11257*sin(q2(t)) - 0.1349*cos(q2(t) + q3(t)) - 0.1351*cos(q2(t) + q3(t) + q4(t)) - 0.028*cos(q2(t)) - 0.0306)*sin(q1(t))],
[-0.0052*sin(q2(t) + q3(t))*cos(q1(t)) - 0.11257*sin(q2(t))*cos(q1(t)) + 0.1349*cos(q2(t) + q3(t))*cos(q1(t)) + 0.1351*cos(q2(t) + q3(t) + q4(t))*cos(q1(t)) + 0.028*cos(q1(t))*cos(q2(t)) + 0.0306*cos(q1(t)) + 0.0452],
[                                                                             0.1349*sin(q2(t) + q3(t)) + 0.1351*sin(q2(t) + q3(t) + q4(t)) + 0.028*sin(q2(t)) + 0.0052*cos(q2(t) + q3(t)) + 0.11257*cos(q2(t)) + 0.119]])

# Jacobian

In [59]:
# 1. Define the state vector
state_q = [q1, q2, q3, q4, q5]

# 2. Linear Jacobian (Jv) - This stays the same
Jl = pos_vector.jacobian(state_q)

# 3. Geometric Rotational Jacobian (Jw) 
omega_G_N = G.ang_vel_in(N).to_matrix(N)
q_dots = [q1.diff(), q2.diff(), q3.diff(), q4.diff(), q5.diff()]
Jr = omega_G_N.jacobian(q_dots)

# 4. Combine to get the full Geometric Jacobian (6x5)
J_geometric = Jl.col_join(Jr)

# 5. Simplify and Create the numerical function
J_sig = sm.simplify(J_geometric)
jac_func = sm.lambdify((q1, q2, q3, q4, q5), J_sig, 'numpy')

# 3. Apply substitution to the simplified Jacobian
J_latex_ready = J_sig.subs(subs_map)
#print(sm.pycode(J_latex_ready)

for i in range(5):
    column_matrix = J_latex_ready.col(i)
    # We simplify each column individually to keep the LaTeX as short as possible
    latex_code = sm.latex(sm.simplify(column_matrix))
    
    #print(f"--- LaTeX for Column J{i+1} ---")
    #print(latex_code)
    #print("\n")

J_sig

ImmutableDenseMatrix([[(0.11257*math.sin(q2) + 0.0052*math.sin(q2 + q3) - 0.028*math.cos(q2) - 0.1349*math.cos(q2 + q3) - 0.1351*math.cos(q2 + q3 + q4) - 0.0306)*math.cos(q1), (0.028*math.sin(q2) + 0.1349*math.sin(q2 + q3) + 0.1351*math.sin(q2 + q3 + q4) + 0.11257*math.cos(q2) + 0.0052*math.cos(q2 + q3))*math.sin(q1), (0.1349*math.sin(q2 + q3) + 0.1351*math.sin(q2 + q3 + q4) + 0.0052*math.cos(q2 + q3))*math.sin(q1), 0.1351*math.sin(q1)*math.sin(q2 + q3 + q4), 0], [(0.11257*math.sin(q2) + 0.0052*math.sin(q2 + q3) - 0.028*math.cos(q2) - 0.1349*math.cos(q2 + q3) - 0.1351*math.cos(q2 + q3 + q4) - 0.0306)*math.sin(q1), -(0.028*math.sin(q2) + 0.1349*math.sin(q2 + q3) + 0.1351*math.sin(q2 + q3 + q4) + 0.11257*math.cos(q2) + 0.0052*math.cos(q2 + q3))*math.cos(q1), -(0.1349*math.sin(q2 + q3) + 0.1351*math.sin(q2 + q3 + q4) + 0.0052*math.cos(q2 + q3))*math.cos(q1), -0.1351*math.sin(q2 + q3 + q4)*math.cos(q1), 0], [0, -0.11257*math.sin(q2) - 0.0052*math.sin(q2 + q3) + 0.028*math.cos(q2) + 0.1349*

Matrix([
[(0.0052*sin(q2(t) + q3(t)) + 0.11257*sin(q2(t)) - 0.1349*cos(q2(t) + q3(t)) - 0.1351*cos(q2(t) + q3(t) + q4(t)) - 0.028*cos(q2(t)) - 0.0306)*cos(q1(t)),  (0.1349*sin(q2(t) + q3(t)) + 0.1351*sin(q2(t) + q3(t) + q4(t)) + 0.028*sin(q2(t)) + 0.0052*cos(q2(t) + q3(t)) + 0.11257*cos(q2(t)))*sin(q1(t)),  (0.1349*sin(q2(t) + q3(t)) + 0.1351*sin(q2(t) + q3(t) + q4(t)) + 0.0052*cos(q2(t) + q3(t)))*sin(q1(t)),  0.1351*sin(q2(t) + q3(t) + q4(t))*sin(q1(t)),                                      0],
[(0.0052*sin(q2(t) + q3(t)) + 0.11257*sin(q2(t)) - 0.1349*cos(q2(t) + q3(t)) - 0.1351*cos(q2(t) + q3(t) + q4(t)) - 0.028*cos(q2(t)) - 0.0306)*sin(q1(t)), -(0.1349*sin(q2(t) + q3(t)) + 0.1351*sin(q2(t) + q3(t) + q4(t)) + 0.028*sin(q2(t)) + 0.0052*cos(q2(t) + q3(t)) + 0.11257*cos(q2(t)))*cos(q1(t)), -(0.1349*sin(q2(t) + q3(t)) + 0.1351*sin(q2(t) + q3(t) + q4(t)) + 0.0052*cos(q2(t) + q3(t)))*cos(q1(t)), -0.1351*sin(q2(t) + q3(t) + q4(t))*cos(q1(t)),                                      0],
[      

Add numpy function to generate the Jacobian


In [54]:
# 2. Define the configurations (joint angles) you found in Task 2.1.2
# Replace these placeholders with your actual numerical IK solutions [q1, q2, q3, q4, q5]
configs = {
    "Pose I":    [-0.9208, 0.6456, -0.8770, 0.2321, 1.5708],
    "Pose II":   [-1.3034, -0.1820, 0.4463, 1.3065, 2.8734],
    "Pose III":  [-2.0000, 1.3108, -1.3106, -1.5708, -2.000],
    "Pose IV-b": [3.141, 0.7352, 1.2885, -1.2358, 0]
}


# 3. Evaluate and print the Jacobian for each configuration
for name, q_values in configs.items():
    # The * unpacks the list into the 5 arguments required by the function
    J_num = jac_func(*q_values)
    
    print(f"--- Numerical Jacobian for {name} ---")
    print(np.round(J_num, 4)) 
    print("\n")


--- Numerical Jacobian for Pose I ---
[[-1.5300000e-01 -6.4500000e-02  2.0500000e-02 -1.0000000e-04
   0.0000000e+00]
 [ 2.0130000e-01 -4.9000000e-02  1.5600000e-02 -1.0000000e-04
   0.0000000e+00]
 [ 0.0000000e+00  2.2220000e-01  2.6760000e-01  1.3510000e-01
   0.0000000e+00]
 [ 0.0000000e+00 -7.4961000e+00 -7.4961000e+00 -7.4961000e+00
   1.4285319e+03]
 [-0.0000000e+00 -1.0000000e+00 -1.0000000e+00 -1.0000000e+00
  -5.2000000e-03]
 [ 1.0000000e+00 -7.4961000e+00 -7.4961000e+00 -7.4961000e+00
   1.4285322e+03]]


--- Numerical Jacobian for Pose II ---
[[-0.0548 -0.271  -0.1691 -0.1303  0.    ]
 [ 0.2    -0.0742 -0.0463 -0.0357  0.    ]
 [ 0.      0.1768  0.1289 -0.      0.    ]
 [ 0.     -0.9643 -0.9643 -0.9643 -0.    ]
 [ 0.     -0.265  -0.265  -0.265   0.    ]
 [ 1.      0.      0.      0.      1.    ]]


--- Numerical Jacobian for Pose III ---
[[ 2.660e-02  6.720e-02  1.181e-01  1.228e-01  0.000e+00]
 [ 5.810e-02 -3.070e-02 -5.400e-02 -5.620e-02  0.000e+00]
 [ 0.000e+00  3.330e-02